# Chapter 01-03 · NumPy: arrays, shapes, and vectorised thinking

**Label:** Optional  |  **Time:** ~50 minutes  |  **Difficulty:** gentle, with one genuinely tricky idea

**Prerequisites:** 01-02, or the equivalent fluency. You should be comfortable with lists,
comprehensions and functions.

**Position in the learning path:** module 01, chapter 3 of 6. Before: **01-02**. After:
**01-04** (pandas).

---

## Why this matters

Every numerical library you will meet - pandas, scikit-learn, PyTorch - is built on NumPy arrays
or on something shaped exactly like them. Learning NumPy is not learning one more library; it is
learning the object that all of them pass around.

There is also a specific reason this chapter exists rather than being folded into pandas: **the
most common numerical bug in machine learning code does not raise an error.** Aggregating over
the wrong axis returns a plausible array of plausible numbers, the model trains, the score
appears, and the result is wrong. The failure lab shows it happening to a standardisation step,
which is exactly where it happens in real projects.

## What you will be able to do

By the end of this chapter you can:

1. **Explain** what an array gives you that a list does not, and measure the speed difference.
2. **Read and predict** the `shape` of any result before running it.
3. **Choose** the right `axis` and state the rule that makes it memorable.
4. **Use** boolean masks and `np.where` to filter and to compute conditionally.
5. **Diagnose** a wrong-axis bug and a view-versus-copy bug, neither of which raises an error.

## Warm-up: retrieve, do not reread

From memory:

1. Why is `def f(x, log=[])` dangerous?
2. What does `b = a` copy?
3. What are the three parts of a comprehension?
4. What does `zip` do when its inputs have different lengths?

<br>

*Answers: (1) the default is created once at definition, so every call shares one list. (2) the
reference - one object, two names. (3) what I want; where it comes from; which ones. (4) stops at
the shortest, silently, unless `strict=True`.*

## The situation

Six sensors again, and one job: **standardise** the temperatures - subtract the mean and divide
by the standard deviation, so the values are centred on zero and measured in "how many typical
deviations from average". Nearly every model in this course needs this at some point, and 04-06
does it properly.

With six numbers you would do it with a comprehension. With six columns and a million rows, that
approach is both slow and - more importantly - very easy to get subtly wrong.

**The question this chapter answers:** how do you do arithmetic on whole tables of numbers at
once, and how do you know you did it to the right dimension?

In [ ]:
import numpy as np

temps_list = [18.5, 24.1, 31.7, 22.0, 27.3, 19.4]
temps = np.array(temps_list)

print("array      :", temps)
print("dtype      :", temps.dtype)      # one type for the whole array
print("shape      :", temps.shape)      # a tuple, always
print("ndim       :", temps.ndim)

### What an array is, and how it differs from a list

An array holds **one type of number, in a fixed shape, in one contiguous block of memory**. A
list holds anything, anywhere.

That constraint is what buys everything else:

- **Arithmetic on the whole thing.** `temps * 2` doubles every element. `temps_list * 2` makes a
  twelve-item list, because for a list `*` means repeat. This is the single most common surprise
  when moving from lists to arrays.
- **Speed**, because the loop happens in compiled code instead of in Python.
- **A shape**, so NumPy can check that operations make sense and can broadcast when they nearly do.

`dtype` is worth noticing on day one. An array of whole numbers has an integer dtype, and integer
arrays do integer things - which is the subject of one of the misconceptions below.

In [ ]:
print("temps * 2         :", temps * 2)
print("temps_list * 2    :", temps_list * 2)
print()
print("mean              :", temps.mean().round(3))
print("standard deviation:", temps.std().round(3))
print("standardised      :", ((temps - temps.mean()) / temps.std()).round(2))

`(temps - temps.mean()) / temps.std()` is the entire standardisation, with no loop. Read it as one
sentence: *take every temperature, subtract the average, divide by the spread.*

The result is in **standard deviations**, not degrees. A value of `1.7` means "1.7 typical
deviations above average". Losing track of that is how people end up reporting a temperature of
1.7 °C when they mean something else entirely - and it is why 04-06 insists on keeping the
original units somewhere.

In [ ]:
import time

n = 1_000_000
xs = list(range(n))
arr = np.arange(n)

start = time.perf_counter()
result_list = []
for x in xs:
    result_list.append(x * 2.0 + 1)
python_seconds = time.perf_counter() - start

start = time.perf_counter()
result_array = arr * 2.0 + 1
numpy_seconds = time.perf_counter() - start

print(f"python loop : {python_seconds:.4f} s")
print(f"numpy       : {numpy_seconds:.5f} s")
print(f"ratio       : {python_seconds / numpy_seconds:.0f}x faster")

The exact ratio depends on your machine and on the operation - anywhere from about ten times to a
few hundred is normal, and it grows with how much arithmetic each element needs.

But speed is the *second* reason to use NumPy. The first is that `arr * 2.0 + 1` says what it
means in one line, and a five-line loop with an accumulator does not. Shorter code with fewer
moving parts has fewer places to hide a mistake - and the mistakes in this chapter are all silent.

## Creating arrays, and shapes

A shape is always a **tuple**, and reading shapes out loud is the most valuable habit in
numerical work.

In [ ]:
print("arange        ", np.arange(5), np.arange(5).shape)
print("zeros         ", np.zeros(3), np.zeros(3).shape)
print("linspace      ", np.linspace(0, 1, 5))
print("2-D from list ", np.array([[1, 2, 3], [4, 5, 6]]).shape, "-> 2 rows, 3 columns")
print("reshape       ", np.arange(6).reshape(2, 3).shape)
print("reshape(-1,2) ", np.arange(6).reshape(-1, 2).shape, "  (-1 means: work it out)")
print()
rng = np.random.default_rng(0)          # always seed - 01-06 and 04-08 explain why
print("random normals", rng.normal(0, 1, 4).round(2))

`reshape(-1, 2)` means "two columns, and figure out how many rows". It is the idiom for reshaping
when you know one dimension and do not want to compute the other - and it will error if the total
number of elements does not divide evenly, which is the good kind of failure.

**The `(n,)` versus `(n, 1)` distinction** trips up everyone eventually. `np.arange(3)` has shape
`(3,)` - a one-dimensional array with no notion of row or column. `np.arange(3).reshape(3, 1)` has
shape `(3, 1)` - a column. scikit-learn asks for a 2-D `X`, which is why you sometimes have to
write `.reshape(-1, 1)` for a single feature, and why the error message says "expected 2D array,
got 1D array instead". You have already met this: in 00-01, `days[["temp_c"]]` used double
brackets to get a DataFrame rather than a Series, for exactly this reason.

## The axis, which is the whole chapter

Here is the six-sensor data as a table: three sites, two sensors each.

### Predict before running

`site_temps` will have shape `(3, 2)`.

1. What shape does `site_temps.mean(axis=0)` return, and what do the numbers mean?
2. What shape does `site_temps.mean(axis=1)` return?
3. What does `site_temps.mean()` with no axis return?

In [ ]:
site_temps = np.array([[18.5, 24.1],      # north
                       [31.7, 22.0],      # south
                       [27.3, 19.4]])     # east

print("shape            ", site_temps.shape)
print("mean(axis=0)     ", site_temps.mean(axis=0).round(2), " shape", site_temps.mean(axis=0).shape)
print("mean(axis=1)     ", site_temps.mean(axis=1).round(2), " shape", site_temps.mean(axis=1).shape)
print("mean()           ", site_temps.mean().round(2), " shape", np.shape(site_temps.mean()))

### The rule

> **`axis=n` is the axis that disappears.**

`site_temps` has shape `(3, 2)`.

- `axis=0` removes the **3**, leaving shape `(2,)`: one number per **column** - the average of
  each sensor slot across sites.
- `axis=1` removes the **2**, leaving shape `(3,)`: one number per **row** - the average per site.
- No axis collapses everything to a single number.

People try to memorise "axis 0 means columns", get it backwards under pressure, and then have to
reason it out from scratch every time. "The axis you name is the one that disappears" is
mechanical, works for three-dimensional arrays and higher, and takes one second to apply.

**And in machine learning, the convention is fixed:** rows are observations, columns are
features. So *"compute something per feature"* - a mean to subtract, a scale to divide by, a
count of missing values - is almost always `axis=0`. If you find yourself wanting `axis=1` for a
preprocessing step, stop and check.

In [ ]:
# Boolean masks: same idea as in 01-01, now on a table.
warm = site_temps > 25
print(warm)
print("\nthe warm values  :", site_temps[warm])
print("how many          :", warm.sum(), "   (True counts as 1)")
print("any warm per site :", warm.any(axis=1))
print("all warm per site :", warm.all(axis=1))

In [ ]:
# np.where: choose element by element. Read as: where(condition, if_true, if_false)
labels = np.where(site_temps > 25, "warm", "ok")
capped = np.where(site_temps > 30, 30.0, site_temps)

print(labels)
print(capped)

`np.where` is the vectorised `if`. It appears constantly: capping outliers, turning a probability
into a decision, building a flag column. You saw it already in 00-01, where
`np.where(is_rainy, "rainy", "sunny")` turned a boolean into labels.

Note that `warm.sum()` counts the `True`s, exactly as summing booleans did in 01-02 - and
`warm.mean()` would give the *proportion*, which is often what you actually want. That one-token
change between count and share is worth remembering; you will use it to ask "what fraction of
rows are missing this field?" more often than almost anything else.

In [ ]:
# Broadcasting: the small array is stretched to fit the big one.
column_means = site_temps.mean(axis=0)            # shape (2,)
centred = site_temps - column_means               # (3,2) - (2,) -> works

print("site_temps shape", site_temps.shape, " column_means shape", column_means.shape)
print(centred.round(2))
print("\ncolumn means of the result:", centred.mean(axis=0).round(10))

**The broadcasting rule:** line the shapes up **from the right**. Dimensions match if they are
equal, or if one of them is 1 (or absent, which counts as 1). The size-1 dimension is repeated.

```
site_temps   (3, 2)
column_means    (2,)     ->  treated as (1, 2), stretched down the rows   -> (3, 2)  OK
```

Trying to subtract the *row* means the same way fails, and it fails loudly:

```
site_temps   (3, 2)
row_means       (3,)     ->  treated as (1, 3), and 3 != 2                -> error
```

That error is a gift. The dangerous case is when the shapes happen to be compatible and you did
not mean them to be - which is the next section.

---

## Failure lab: the standardisation that ran perfectly and was wrong

Three features on wildly different scales - a temperature, a percentage and a count - and a
target that depends on all three. We standardise the features, which is routine.

There are two ways to write it. Both run. Both return an array of the right shape full of
sensible-looking numbers.

**Predict before running:** which one is wrong, and how would you find out if nobody told you?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(5)
n_rows = 300
X = np.column_stack([rng.normal(20, 5, n_rows),        # temperature, roughly 20
                     rng.normal(60, 20, n_rows),       # humidity, roughly 60
                     rng.normal(1000, 300, n_rows)])   # a count, roughly 1000
y = 2.0 * X[:, 0] - 0.5 * X[:, 1] + 0.01 * X[:, 2] + rng.normal(0, 3, n_rows)   # SYNTHETIC

per_feature = (X - X.mean(axis=0)) / X.std(axis=0)
per_row = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

print("X shape          ", X.shape)
print("per_feature shape", per_feature.shape)
print("per_row shape    ", per_row.shape, "  <- identical, and no warning from either")

Same shape, no error, no warning. Now the one-line check that tells them apart.

In [ ]:
print("per_feature - mean of each column:", per_feature.mean(axis=0).round(6))
print("per_row     - mean of each column:", per_row.mean(axis=0).round(3))
print()
print("per_feature - mean of each row (first 3):", per_feature.mean(axis=1)[:3].round(3))
print("per_row     - mean of each row (first 3):", per_row.mean(axis=1)[:3].round(6))

There it is. `per_feature` has **columns** centred on zero, which is what standardising features
means. `per_row` has **rows** centred on zero: every observation has been rescaled against its own
three numbers - a temperature, a humidity and a count averaged together, which is not a
meaningful quantity in any universe.

Now the consequence.

In [ ]:
def score(features, name):
    train, test = slice(0, 200), slice(200, None)
    model = LinearRegression().fit(features[train], y[train])
    print(f"  {name:<32} MAE {mean_absolute_error(y[test], model.predict(features[test])):.2f}")

print("predicting y from three features:")
score(per_feature, "standardised per feature")
score(per_row, "standardised per row (the bug)")
score(X, "raw, not standardised at all")
print(f"  {'baseline: always the mean':<32} MAE "
      f"{mean_absolute_error(y[200:], [y[:200].mean()] * 100):.2f}")

### Diagnosis

**2.16 against 7.91.** The bug more than triples the error, and everything about the run looked
normal: correct shapes, no warnings, a model that trained and produced predictions.

**Why it destroys the information.** Standardising per feature keeps each column's *ordering* -
the hottest day is still the highest value in the temperature column, just measured in deviations
instead of degrees. Standardising per row asks "how does this row's temperature compare with this
row's humidity and this row's count?", which mixes three unrelated units into one meaningless
scale and throws away the comparison the model needed.

**How you find it without being told: check the thing you claimed to do.** You said you
standardised the features, so the features should have mean 0 and standard deviation 1:

```python
assert np.allclose(per_feature.mean(axis=0), 0)
assert np.allclose(per_feature.std(axis=0), 1)
```

Two lines. They would have failed instantly on `per_row`.

**Notice also the third row of that output.** Raw, unstandardised features score **exactly the
same 2.16** as correct standardisation. Ordinary linear regression does not care about feature
scale at all - it simply learns different coefficients. Standardising matters for *regularised*
models (05-09), *distance-based* models (06-10, 08-02) and *neural networks* (module 10), and it
is worth knowing that the step you are debugging was optional for this particular model. Not every
preprocessing step earns its place; 04-06 makes you justify each one.

### Remedies

| Remedy | What it catches | Cost |
|---|---|---|
| Print `.shape` at every step | Mismatches, accidental broadcasts | One line, and it is the highest-value line in numerical code |
| Assert the property you intended | This exact bug | Two lines per transformation |
| Remember "rows are observations, columns are features" | Reaching for `axis=1` in preprocessing | Free |
| Use `sklearn`'s `StandardScaler` in a pipeline | The axis, *and* fitting on the wrong rows | 04-06, 04-07 - and it is the real answer |

The last row is the honest one. You will not hand-write standardisation in production; you will
use `StandardScaler` inside a `Pipeline`, which cannot get the axis wrong and cannot leak the test
set. This chapter teaches the mechanism so the library is not magic - not so you reimplement it.

### The second silent bug: a slice shares memory

In [ ]:
original = np.arange(6)
window = original[:3]        # a VIEW - not a copy
window[0] = 99

print("original after editing the slice:", original, "  <- changed")

original = np.arange(6)
safe = original[:3].copy()   # an explicit copy
safe[0] = 99
print("original after editing a copy   :", original, "  <- unchanged")

Slicing a NumPy array gives a **view**: a window onto the same memory, not a new array. Writing
through the view writes through to the original.

This is deliberate and valuable - it means slicing a gigabyte array costs nothing - but it is
exactly the aliasing bug from 01-02 wearing different clothes, and it is why `.copy()` exists.

**Where it bites in this course:** you split `X` into train and test with slices, then modify the
training slice in place - clipping outliers, filling missing values - and you have silently
modified the test set too. That is leakage created by a slice, and 04-05 has an example.

**The safe habits:** avoid in-place modification of arrays you sliced from something else, or take
an explicit `.copy()`. And note that *fancy* indexing (`arr[[0, 2, 4]]` or `arr[mask]`) always
returns a copy, while plain slicing returns a view - which is an inconsistency you simply have to
know.

## Common misconceptions

**"An integer array will hold whatever I put in it."**
It will truncate. `np.array([1, 2, 3])` has an integer dtype, and assigning `2.7` into it stores
`2`. Worse, `arr /= 2` on an integer array raises, while `arr //= 2` silently floors. Create
arrays as floats when they hold measurements: `np.array([1, 2, 3], dtype=float)`.

**"`np.mean` and `arr.mean()` are different things."**
They are the same function. Most NumPy operations exist both as a function and as a method; use
whichever reads better.

**"NaN behaves like a missing value."**
`np.nan` propagates: any arithmetic involving it gives `nan`, so one bad element makes a whole
mean `nan`. And `np.nan == np.nan` is `False`, so you cannot test for it with `==` - use
`np.isnan`. The `nan`-aware functions (`np.nanmean`, `np.nansum`) skip them, which is sometimes
right and sometimes hides a data problem you should have seen. 02-04.

**"Bigger arrays are always faster in NumPy than in Python."**
For very small arrays the fixed overhead of a NumPy call can make it *slower* than a plain loop.
NumPy wins on volume, not on principle. Measure before optimising anything.

**"If the shapes broadcast, the operation is correct."**
Broadcasting is a rule about shapes, not about meaning. `(3, 1)` against `(1, 3)` produces a
`(3, 3)` array quite happily, and if you expected 3 numbers and got 9, nothing told you. When an
array is unexpectedly large, suspect a broadcast.

**"I should vectorise everything."**
Readability first. A loop over ten configurations is clearer than a clever reshape, and it runs in
microseconds either way. Vectorise the inner loop over a million rows, not the outer loop over
five experiments.

---

## Exercises

Solutions: `solutions/01_python_bridge/01-03_numpy_solutions.ipynb`.

### Quick understanding

**E1 (define).** State the axis rule in one sentence, and apply it: an array of shape `(50, 4)`,
what shape does `.sum(axis=1)` give?

**E2 (explain).** Why does `temps * 2` double the values but `temps_list * 2` lengthen the list?

**E3 (explain).** What is the difference between a view and a copy, and which one does `arr[:5]`
give you?

### Hand calculation

**E4 (calculate).** For `M = np.array([[1, 2], [3, 4], [5, 6]])`, write down without running:
(a) `M.shape`, (b) `M.sum(axis=0)`, (c) `M.sum(axis=1)`, (d) `M.sum()`, (e) the shape of
`M - M.mean(axis=0)`.

**E5 (calculate).** Say for each pair whether broadcasting succeeds, and the resulting shape:
(a) `(4, 3)` and `(3,)`, (b) `(4, 3)` and `(4,)`, (c) `(4, 3)` and `(4, 1)`, (d) `(4, 1)` and
`(1, 3)`, (e) `(4, 3)` and `(1, 3)`. Then say which of these is the dangerous one and why.

### Coding

**E6 (code).** Write `standardise(X)` returning the per-feature standardised array, and add two
`assert` statements that would have caught the failure lab's bug. Test it on the chapter's `X`.

**E7 (code).** Given `y_true` and `y_pred` arrays, compute MAE with NumPy in one line, without a
loop and without scikit-learn. Then compute the *index* of the single worst prediction and print
the actual and predicted values there.

### Interpretation

**E8 (interpret).** In the failure lab, raw features and correctly standardised features gave
identical MAE. Explain why, and name two model families where the answer would have been very
different.

### Debugging

**E9 (diagnose).** This is meant to compute each student's average across the four tests, and
then how far each score sits above that student's own average. It runs without error and the
numbers are wrong.

```python
scores = np.array([[70, 80, 90, 100], [60, 65, 70, 75], [88, 92, 96, 100]])
averages = scores.mean(axis=0)
above = scores - averages
```

Say what `averages` actually contains, what `above` contains, and give the fix.

### Exam and interview reasoning

**E10 (defend).** *"Why is NumPy faster than a Python loop?"* Answer in four sentences without
saying "because it is written in C" and nothing else.

**E11 (design).** You have a `(100000, 50)` feature matrix and need to drop every row containing a
`NaN`, then standardise the remainder. Write the three lines, in order, and say why that order is
the only correct one.

### Transfer to a different situation

**E12 (design).** An image is loaded as an array of shape `(256, 256, 3)` - height, width, colour
channels. Say what each of these computes, and give its shape: (a) `img.mean(axis=(0, 1))`,
(b) `img.mean(axis=2)`, (c) `img.max()`. Which one would you use to convert the image to
greyscale, and which to find the average colour?

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to a colleague what "standardising a feature" does
and why anyone would bother. No formulas.

### Optional challenge

**E14 (code).** Implement `pairwise_distances(A, B)` returning the matrix of Euclidean distances
between every row of `A` (shape `(n, d)`) and every row of `B` (shape `(m, d)`), using
broadcasting and no loops. Check it against a double loop on small inputs. Then say what happens
to memory when `n` and `m` are 50,000 - and what that tells you about when a clever
broadcasting trick is the wrong answer.

In [ ]:
# Your workspace. Still in memory: temps, site_temps, X, y, per_feature, per_row, rng.

## Mastery check

Without scrolling up, can you:

- [ ] State the axis rule and apply it to a `(50, 4)` array? *(If not: "The axis".)*
- [ ] Say the broadcasting rule and which direction shapes are aligned from? *(If not:
      "Broadcasting".)*
- [ ] Name the two-line check that catches a wrong-axis standardisation? *(If not: "Failure lab".)*
- [ ] Say whether `arr[:5]` shares memory with `arr`? *(If not: "The second silent bug".)*
- [ ] Explain why an integer array is a hazard for measurements? *(If not: "Misconceptions".)*

## What should now feel instinctive

1. **Print the shape.** Before and after anything that reshapes, aggregates or broadcasts.
2. **"The axis you name is the one that disappears."**
3. **Rows are observations, columns are features** - so preprocessing is `axis=0`.
4. **Assert the property you claimed to create.** "I standardised the features" is a testable
   statement; test it.
5. **Slices share memory.** If you are about to modify one, take a copy or do not modify it.

## Flashcards

| Question | Answer |
|---|---|
| The axis rule | `axis=n` is the axis that disappears |
| Shape after `(3, 4).mean(axis=0)` | `(4,)` - one value per column |
| The broadcasting rule | Align shapes from the right; dimensions must be equal, or one of them 1 |
| `arr * 2` vs `list * 2` | Doubles every element vs repeats the list |
| What does `reshape(-1, 2)` mean? | Two columns; work out the number of rows |
| `(n,)` vs `(n, 1)` | A flat array vs a column. scikit-learn wants 2-D `X` |
| View or copy for `arr[:5]`? | A view - it shares memory. Fancy indexing gives a copy |
| Why can't you test for NaN with `==`? | `np.nan == np.nan` is `False`. Use `np.isnan` |
| Two lines that catch a wrong-axis standardisation | `assert np.allclose(Z.mean(axis=0), 0)` and the same for `.std(axis=0)` being 1 |
| When is NumPy *not* faster? | On very small arrays, where the call overhead dominates |

## Next

**01-04 · pandas I: loading, selecting, filtering, dtypes.**

NumPy gives you a block of numbers with a shape. Real data has column names, mixed types, missing
values and an index - and that is what pandas adds on top of exactly the arrays you have just
been using. Everything here still applies: pandas columns *are* NumPy arrays, `axis=0` still means
the same thing, and views versus copies returns as the most-discussed warning in the library.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).